# AB1 · Country groups head to head (A vs B)

> **Question AB1:** Do listed companies in EU countries earn a different return on assets than companies in non-EU countries, once sector, size and year are taken into account?

| | |
|---|---|
| **Why "A vs B"** | The earlier notebooks relate ROA to a continuous index (H0). Decision-makers often think in two groups ("EU or not", "rich or poor"). A two-group comparison answers that directly and gives one number: the gap in ROA between the groups. |
| **What this is not** | A real A/B test randomly assigns the treatment. Countries cannot be assigned to the EU, so the groups differ in many other ways (development, institutions, market structure). The gap is a **descriptive difference**, not the effect of EU membership. |
| **Prediction** | None on direction (two-sided). EU firms may earn less (H0 found lower ROA where countries are more developed, and the EU is mostly developed) or more. |
| **Data** | Yahoo company-years, 34 countries (21 EU, 13 non-EU). ESEF cannot replicate this: it contains only EU filers. |
| **Primary test (fixed before looking at the result)** | `ROA (pp) ~ EU dummy + sector + size rank + fiscal year`; the test is the coefficient on the EU dummy. Errors clustered by country, wild-bootstrap p-value (21 vs 13 countries). |
| **Secondary comparisons (not part of the verdict)** | (a) High vs low development half of the countries. (b) High vs low institutions half. (c) EU vs non-EU **holding development constant**: does the gap survive once development is controlled? (d) The EU gap by firm size. |
| **Verdict rule** | The project rule (`utils.verdict`, direction not predicted): *supported* if wild p < 0.05 and the sign holds in at least 80% of the robustness checks; *partly supported* if p < 0.10 or robustness is lower; otherwise *not supported*. |

**Honest framing.** Only 34 countries are independent observations, so the p-value comes from a wild cluster bootstrap and is cross-checked with a permutation test that shuffles the EU label across countries. This comparison is **outside the 15-test multiple-testing family** of `1.0_hypotheses.ipynb` (its result is stored as `results/AB1.json`, not `H*.json`); if it were added to the family, the other q-values would shift slightly. Association, not causation.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils, compare
from utils import CONTROLS, fit_cluster, wild_cluster, bh

con = prep.connect()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile = country["dim"], country["profile"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
EU, NON_EU = list(dim.index[dim.is_eu]), list(dim.index[~dim.is_eu])
print(f"{len(dim)} countries ({len(EU)} EU, {len(NON_EU)} non-EU) | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

34 countries (21 EU, 13 non-EU) | 10,220 Yahoo company-years of 2,590 companies


## 1 · The two groups
Country-level ROA after removing sector, size and year effects (so a country is not 'high' just because of its sector mix).

In [2]:
cm = compare._country_means(fy.dropna(subset=["roa_pp"]), "roa_pp", CONTROLS)
tab = pd.DataFrame({"country": names, "group": pd.Series(np.where(dim.is_eu, "EU", "non-EU"), index=dim.index), "adj. ROA (pp)": cm, "firm-years": fy.groupby("iso3").size()}).dropna()
tab = tab.sort_values("adj. ROA (pp)")
fig = px.strip(tab, x="adj. ROA (pp)", y="group", color="group", hover_name="country", color_discrete_map={"EU": "#1565c0", "non-EU": "#e53935"}, title="ROA by country (adjusted for sector, size, year), by group")
for g, v in tab.groupby("group")["adj. ROA (pp)"].mean().items():
    fig.add_vline(x=v, line_dash="dot", line_color="grey", annotation_text=f"{g} mean", annotation_position="top")
fig.update_traces(marker_size=11); fig.update_layout(height=340, showlegend=False); fig.show()
print(tab.groupby("group").agg(countries=("country", "size"), mean=("adj. ROA (pp)", "mean"), sd=("adj. ROA (pp)", "std"), firm_years=("firm-years", "sum")).round(2).to_string())

        countries   mean    sd  firm_years
group                                     
EU             21 -0.910 1.290        5121
non-EU         13  1.120 1.770        5099


## 2 · Primary test

In [3]:
primary = compare.compare_groups(fy, NON_EU, EU, B=1999)       # difference = EU minus non-EU
print(f"EU minus non-EU: {primary['diff']:+.2f} pp ROA (95% CI {primary['ci_low']:+.2f} to {primary['ci_high']:+.2f})")
print(f"p (wild cluster bootstrap) = {primary['p_wild']:.4f} | p (cluster-robust) = {primary['p_cluster']:.4f} | p (country permutation) = {primary['p_perm']:.4f}")
print(f"Raw means (no controls): non-EU {primary['raw_mean_a']:.2f} pp, EU {primary['raw_mean_b']:.2f} pp | {primary['countries_a']} non-EU and {primary['countries_b']} EU countries, {primary['firm_years_a'] + primary['firm_years_b']:,} company-years")
print("Warnings:", primary["warnings"] or "none")

# what the permutation test does: shuffle the EU label over the 34 countries and see how often a gap this large appears
rng = np.random.default_rng(0)
m = primary["country_means"]; lab = m.index.isin(EU)
null = np.array([(lambda p: m.values[p].mean() - m.values[~p].mean())(rng.permutation(lab)) for _ in range(20000)])
obs = m[lab].mean() - m[~lab].mean()
fig = px.histogram(x=null, nbins=60, title="Gap in country-mean ROA if the EU label were assigned at random (20,000 shuffles)", labels={"x": "EU minus non-EU (pp)"})
fig.add_vline(x=obs, line_color="#e53935", annotation_text=f"observed {obs:+.2f}"); fig.update_layout(height=340, showlegend=False); fig.show()

EU minus non-EU: -2.28 pp ROA (95% CI -3.41 to -1.15)
p (wild cluster bootstrap) = 0.0005 | p (cluster-robust) = 0.0004 | p (country permutation) = 0.0006
Raw means (no controls): non-EU 5.24 pp, EU 2.95 pp | 13 non-EU and 21 EU countries, 10,220 company-years
Warnings: ['a country with fewer than 50 firm-years is included']


## 3 · Secondary comparisons

In [4]:
lo_d, hi_d = compare.median_split(country["dev_idx"]); lo_i, hi_i = compare.median_split(country["inst_idx"])
rows = []
for label, a, b in [("EU vs non-EU", NON_EU, EU), ("high vs low development", lo_d, hi_d), ("high vs low institutions", lo_i, hi_i)]:
    r = compare.compare_groups(fy, a, b, B=999)
    rows.append({"comparison (B minus A)": label, "diff (pp)": r["diff"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "p (perm)": r["p_perm"], "countries A/B": f"{r['countries_a']}/{r['countries_b']}"})
sec = pd.DataFrame(rows).set_index("comparison (B minus A)")
sec["q (BH)"] = bh(sec["p (wild)"])
print("Overlap between the splits: the development split puts", len(set(hi_d) & set(EU)), "of", len(hi_d), "high-development countries in the EU.")
sec.round(4)

Overlap between the splits: the development split puts 11 of 17 high-development countries in the EU.


,diff (pp),ci_low,ci_high,p (wild),p (perm),countries A/B,q (BH)
comparison (B minus A),,,,,,,
EU vs non-EU,-2.284,-3.412,-1.155,0.001,0.001,13/21,0.003
high vs low development,-1.655,-2.930,-0.379,0.019,0.020,17/17,0.019
high vs low institutions,-2.014,-3.216,-0.813,0.003,0.012,17/17,0.004


### EU vs non-EU holding development constant
The EU dummy plus the development index in the same model. If the EU gap is just 'the EU is developed', it should shrink toward zero.

In [5]:
fy["eu"] = fy.iso3.isin(EU).astype(float)
ctrl = wild_cluster("eu", fy, outcome="roa_pp", extra="dev_z", B=1999)
print(f"EU gap holding development fixed: {ctrl['coef']:+.2f} pp (95% CI {ctrl['ci_low']:+.2f} to {ctrl['ci_high']:+.2f}), wild p = {ctrl['p_wild']:.3f}  [without the control: {primary['diff']:+.2f} pp]")
print(f"Share of the raw EU gap that development accounts for: {1 - ctrl['coef'] / primary['diff']:.0%}")

EU gap holding development fixed: -1.61 pp (95% CI -2.72 to -0.49), wild p = 0.002  [without the control: -2.28 pp]
Share of the raw EU gap that development accounts for: 30%


### The EU gap by firm size
Size tercile within country (H13 found the development effect lives in small firms).

In [6]:
fy["size_t"] = pd.cut(fy.size_rank, [0, 1/3, 2/3, 1.0001], labels=["small", "medium", "large"], include_lowest=True)
rows = []
for lab, g in fy.groupby("size_t", observed=True):
    r = compare.compare_groups(g, NON_EU, EU, B=999, controls="C(sector) + C(fiscal_year)", perm=False)
    rows.append({"size": lab, "EU minus non-EU (pp)": r["diff"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "firm-years": r["firm_years_a"] + r["firm_years_b"]})
by_size = pd.DataFrame(rows).set_index("size")
fig = px.scatter(by_size.reset_index(), x="EU minus non-EU (pp)", y="size", error_x=by_size.ci_high - by_size["EU minus non-EU (pp)"], error_x_minus=by_size["EU minus non-EU (pp)"] - by_size.ci_low, title="EU gap in ROA by company size")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=300, yaxis_autorange="reversed"); fig.show()
by_size.round(3)

,EU minus non-EU (pp),ci_low,ci_high,p (wild),firm-years
size,,,,,
small,-5.360,-7.877,-2.844,0.001,3373
medium,-1.806,-2.988,-0.625,0.010,3402
large,-0.151,-0.878,0.576,0.695,3445


## 4 · Robustness
The EU gap re-estimated in different samples, and with each of the 34 countries dropped in turn (does one country create the gap?).

In [7]:
SAMPLES = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "excluding Ireland and Luxembourg (tax hubs)": lambda d: d[~d.iso3.isin(["IRL", "LUX"])],
}
rows = []
for s, f in SAMPLES.items():
    d = f(fy); r = compare.compare_groups(d, [c for c in NON_EU if c in set(d.iso3)], [c for c in EU if c in set(d.iso3)], B=499)
    rows.append({"sample": s, "diff": r["diff"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "countries": r["countries_a"] + r["countries_b"], "firm-years": r["firm_years_a"] + r["firm_years_b"]})
rob = pd.DataFrame(rows).set_index("sample")
loo = []
for c in dim.index:
    d = fy[fy.iso3 != c].dropna(subset=["roa_pp"]); d = d.assign(eu=d.iso3.isin(EU).astype(float))
    loo.append({"dropped": names[c], "diff": fit_cluster(f"roa_pp ~ eu + {CONTROLS}", d).params["eu"]})
loo = pd.DataFrame(loo).set_index("dropped").sort_values("diff")
expected_sign = np.sign(primary["diff"])
robust_share = float(np.mean(np.r_[np.sign(rob["diff"]) == expected_sign, np.sign(loo["diff"]) == expected_sign]))
fig = px.scatter(rob.reset_index(), x="diff", y="sample", error_x=rob.ci_high - rob["diff"], error_x_minus=rob["diff"] - rob.ci_low, title="EU minus non-EU ROA gap (pp) in different samples")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=380, yaxis_autorange="reversed"); fig.show()
print(f"Same sign as the primary result in {robust_share:.0%} of {len(rob) + len(loo)} checks. Leave-one-country-out range: {loo['diff'].min():+.2f} to {loo['diff'].max():+.2f} pp (dropping {loo['diff'].idxmin()} / {loo['diff'].idxmax()}).")
rob.round(3)

Same sign as the primary result in 100% of 40 checks. Leave-one-country-out range: -2.51 to -2.00 pp (dropping Singapore / India).


,diff,ci_low,ci_high,p (wild),countries,firm-years
sample,,,,,,
all firms,-2.284,-3.412,-1.155,0.002,34,10220
excluding financials & real estate,-2.578,-3.964,-1.192,0.002,34,8063
excluding small-sample countries,-2.316,-3.473,-1.159,0.002,29,9787
"excluding USA, China, Japan",-1.908,-3.088,-0.729,0.004,31,9023
fiscal years 2023 and later,-2.372,-3.541,-1.203,0.002,34,7603
excluding Ireland and Luxembourg (tax hubs),-2.260,-3.402,-1.118,0.002,32,10009


## 5 · Verdict

In [8]:
v = utils.verdict(primary["diff"], primary["p_wild"], 0, robust_share, primary["countries_a"] + primary["countries_b"])
lines = [
 f"Primary: EU minus non-EU = {primary['diff']:+.2f} pp ROA (95% CI {primary['ci_low']:+.2f} to {primary['ci_high']:+.2f}), wild p = {primary['p_wild']:.4f}, permutation p = {primary['p_perm']:.4f}, {primary['countries_a'] + primary['countries_b']} countries, {primary['firm_years_a'] + primary['firm_years_b']:,} company-years.",
 f"Holding development constant the gap is {ctrl['coef']:+.2f} pp (wild p = {ctrl['p_wild']:.3f}). High vs low development: {sec.iloc[1]['diff (pp)']:+.2f} pp (p = {sec.iloc[1]['p (wild)']:.3f}); high vs low institutions: {sec.iloc[2]['diff (pp)']:+.2f} pp (p = {sec.iloc[2]['p (wild)']:.3f}).",
 "By size: " + "; ".join(f"{i} {r['EU minus non-EU (pp)']:+.2f}" for i, r in by_size.iterrows()) + " pp.",
 f"Robustness: same sign in {robust_share:.0%} of {len(rob) + len(loo)} checks (six samples and 34 leave-one-out runs). ESEF replication not possible (EU filers only).",
 f"VERDICT AB1: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("AB1", id="AB1", title="EU vs non-EU: gap in company ROA", statement="Companies in EU countries earn a different ROA than those in non-EU countries (two-sided).",
                  expected_sign=0, effect=primary["diff"], ci_low=primary["ci_low"], ci_high=primary["ci_high"], effect_unit="pp ROA, EU minus non-EU",
                  p_primary=primary["p_wild"], p_method="wild cluster bootstrap", p_permutation=primary["p_perm"], n=primary["firm_years_a"] + primary["firm_years_b"],
                  clusters=primary["countries_a"] + primary["countries_b"], robust_share=robust_share, replication_coef=None, replication_p=None, replication_same_sign=None,
                  verdict=v, notebook="2.1_country_groups_a_vs_b.ipynb", notes=lines)

Primary: EU minus non-EU = -2.28 pp ROA (95% CI -3.41 to -1.15), wild p = 0.0005, permutation p = 0.0006, 34 countries, 10,220 company-years.
Holding development constant the gap is -1.61 pp (wild p = 0.002). High vs low development: -1.65 pp (p = 0.019); high vs low institutions: -2.01 pp (p = 0.003).
By size: small -5.36; medium -1.81; large -0.15 pp.
Robustness: same sign in 100% of 40 checks (six samples and 34 leave-one-out runs). ESEF replication not possible (EU filers only).
VERDICT AB1: SUPPORTED


### Limits
- **Not an experiment.** EU membership is not randomly assigned; the gap mixes the effect of membership with development, institutions, market structure and which firms are listed.
- 21 vs 13 countries: the p-value is only as good as the bootstrap with that few clusters, which is why the permutation test is shown beside it.
- Controls are sector, size rank and year; unobserved firm differences (ownership, leverage, accounting standards: IFRS in the EU vs US GAAP) remain.
- Yahoo coverage differs by country (a few countries have under 100 firm-years); no ESEF replication is possible.
- Association, not causation.

In [9]:
con.close()